<a id="top"></a>
# Fitting Binary Lenses

<hr style="border: 1.5pt solid #a859e4; width: 100%; margin-top: -10px;">

<a href="https://colab.research.google.com/github/rges-pit/data-challenge-notebooks/blob/main/AAS%20Workshop/Session%20C:%20Binary%20Lens/Fitting_Binary_Lenses.ipynb" target="_blank" rel="noopener"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Learning Goals
By the end of this notebook, you will be able to:
- Describe how binary-lens parameters ($s$, $q$, $\alpha$, $\rho$) control caustic topology and magnification structure.
- Construct binary-lens models using [MulensModel](https://github.com/rpoleski/MulensModel?tab=readme-ov-file) and fit them to a simulated Roman light curve using MCMC minimization.
- Obtain a good initial guess for planetary parameters in a light curve through visual inspection.
- Perform a grid search to find the best initial planetary parameters.
- Learn about standard practices in modelling binary-lens events.

## Introduction

This tutorial teaches binary-lens fitting techniques and is primarily intended for participants in the 2026 Sagan Summer Workshop. You will explore different methods for fitting binary-lens light curves and develop an understanding of current modeling practices in the field, why binary-lens events are challenging to fit, and different ways to overcome those challenges.

The workflow for this notebook consists of:
* [Environment Setup](#Environment-Setup)
  - [Imports](#Imports)
  - [Notebook Assets](#notebook-assets)
* [A Brief Introduction to Binary Lens Microlensing](#a-brief-review-of-binary-lens-microlensing)
* [The Binary Lens Model](#the-binary-lens-model)
* [Fitting a Binary Lens Model](#fitting-a-binary-lens-model)
* [Additional Resources](#additional-resources)
* [About the Notebook](#about-this-notebook)


## Environment Setup

If you are new to Jupyter notebooks, the [Getting Started guide](https://medium.com/codingthesmartway-com-blog/getting-started-with-jupyter-notebook-for-python-4e7082bd5d46) offers a quick primer before you dive into this workflow.

<!-- COLAB-ONLY -->
> ### Running on Google Colab
> 1. Open this notebook via the **Open in Colab** button in the workshop README or by pasting the GitHub URL into [https://colab.research.google.com](https://colab.research.google.com).
> 1. Ensure you are signed into a Google account so Colab can save to your Drive.
> 1. Use the `Runtime > Change runtime type` menu to select a GPU **off**, since this workflow is CPU bound, or select a local runtime.
> 1. Remember to download your results or save a copy to Drive; Colab sessions are ephemeral.


In [ ]:
# COLAB-ONLY
%pip install --quiet VBMicrolensing emcee corner multiprocess MulensModel ipympl

# Colab widget bug work-around
get_ipython().kernel.do_shutdown(restart=True)

In [ ]:
# COLAB-ONLY
from google.colab import output
output.enable_custom_widget_manager()

Please **save this notebook in a space you control** (GitHub fork, local repo, Nexus team storage, or Google Drive) so you can return to it later. Hosted environments such as Colab will not auto-save changes to the canonical repository.

### Imports

In [ ]:
#Import libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.optimize as op
import emcee
from multiprocess import Pool
from multiprocess import cpu_count
import corner
from scipy.optimize import lsq_linear
import ipywidgets as widgets
from ipywidgets import FloatSlider, FloatLogSlider
from IPython.display import display
import MulensModel as mm
import matplotlib.lines as mlines
import VBMicrolensing as vbm

try:
    from google.colab import sheets  # this will only work if you are running on Colab
    %matplotlib ipympl  
except:
    %matplotlib inline

import sys
import shutil
import urllib.request
from pathlib import Path
import pickle

### Notebook Assets
This session depends on a few small helper files (e.g. `microlens_emcee.py`, data tables, and static plots). The cell below makes sure those assets are available whether you run locally, on Colab, or on the Roman Research Nexus. Existing local copies are left untouched; missing files are fetched directly from GitHub.


In [ ]:
ASSET_SOURCES = {
    Path("microlens_emcee.py"): "https://raw.githubusercontent.com/rges-pit/sagan_session1/blob/main/Binary-Lens-Fitting/microlens_emcee.py",
    Path("dc1_186_W149.txt"): "https://raw.githubusercontent.com/rges-pit/sagan_session1/blob/main/Binary-Lens-Fitting/dc1_186_W149.txt",
    Path("dc1_186_params.txt"): "https://raw.githubusercontent.com/rges-pit/sagan_session1/blob/main/Binary-Lens-Fitting/dc1_186_params.txt",
    Path("Plots/binary_parameters.png"): "https://raw.githubusercontent.com/rges-pit/sagan_session1/blob/main/Binary-Lens-Fitting/Plots/binary_parameters.png",
    Path("Plots/caustic_structures.png"): "https://raw.githubusercontent.com/rges-pit/sagan_session1/blob/main/Binary-Lens-Fitting/Plots/caustic_structures.png",
    Path("Plots/caustic_topology.png"): "https://raw.githubusercontent.com/rges-pit/sagan_session1/blob/main/Binary-Lens-Fitting/Plots/caustic_topology.png",
    Path("Plots/dc1_071_binlens.png"): "https://raw.githubusercontent.com/rges-pit/sagan_session1/blob/main/Binary-Lens-Fitting/Plots/dc1_071_binlens.png",
    Path("informed_guess_mcmc_chains.npy"): "https://raw.githubusercontent.com/rges-pit/sagan_session1/blob/main/Binary-Lens-Fitting/informed_guess_mcmc_chains.npy",
    Path("informed_guess_mcmc_posteriors.npy"): "https://raw.githubusercontent.com/rges-pit/sagan_session1/blob/main/Binary-Lens-Fitting/informed_guess_mcmc_posteriors.npy",
    Path("informed_guess_mcmc_lnprob.npy"): "https://raw.githubusercontent.com/rges-pit/sagan_session1/blob/main/Binary-Lens-Fitting/informed_guess_mcmc_lnprob.npy",
    Path("informed_guess_mcmc_metadata.json"): "https://raw.githubusercontent.com/rges-pit/sagan_session1/blob/main/Binary-Lens-Fitting/informed_guess_mcmc_metadata.json",
    Path("grid_search_results.pkl"): "https://raw.githubusercontent.com/rges-pit/sagan_session1/blob/main/Binary-Lens-Fitting/grid_search_results.pkl",
    Path("grid_search_mcmc_chains.npy"): "https://raw.githubusercontent.com/rges-pit/sagan_session1/blob/main/Binary-Lens-Fitting/grid_search_mcmc_chains.npy",
    Path("grid_search_mcmc_posteriors.npy"): "https://raw.githubusercontent.com/rges-pit/sagan_session1/blob/main/Binary-Lens-Fitting/grid_search_mcmc_posteriors.npy",
    Path("grid_search_mcmc_lnprob.npy"): "https://raw.githubusercontent.com/rges-pit/sagan_session1/blob/main/Binary-Lens-Fitting/grid_search_mcmc_lnprob.npy",
    Path("grid_search_mcmc_metadata.json"): "https://raw.githubusercontent.com/rges-pit/sagan_session1/blob/main/Binary-Lens-Fitting/grid_search_mcmc_metadata.json",
    
}


def ensure_asset(rel_path: Path, url: str) -> None:
    """Download asset if it is missing from the working directory."""
    rel_path = Path(rel_path)
    if rel_path.exists():
        print(f"✔ {rel_path} already present")
        return

    rel_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"⬇️  Fetching {rel_path} ...")
    with urllib.request.urlopen(url) as response, open(rel_path, "wb") as outfile:
        shutil.copyfileobj(response, outfile)


for path, url in ASSET_SOURCES.items():
    ensure_asset(path, url)

module_dir = Path("microlens_emcee.py").resolve().parent
if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))
    print(f"🔗 Added {module_dir} to sys.path")

## A Brief Review of Binary Lens Microlensing

Finding the magnification of a binary lens event at any point in the source's relative trajectory requires solving the lens equation for the system. The lens equation for a binary point-mass lens maps the angular position of the source to the angular image positions for a given lens configuration. It can be written in complex coordinates as,

$$z_{\mathrm{s}}=z-\frac{1}{1+q}\left(\frac{1}{\bar{z}-\bar{z}_1}+\frac{q}{\bar{z}-\bar{z}_2}\right)$$


where $z_{\mathrm{s}}$ is the position of the source in complex coordinates ($z = x + iy$), $z$ is the position of the image, $z_1$ and $z_2$ are the positions of the lens components, and $q = M_2/M_1$ is the mass ratio between the lens components. All angular positions are scaled to the angular Einstein ring radius ($\theta_E$).

The magnification of an image in this formalism is given by the inverse of the Jacobian of the mapping above evaluated at the image position,

$$\mu = \left.\frac{1}{\operatorname{det} J}\right|_{z=z_j}, \operatorname{det} J \equiv \frac{\partial\left(x_1, x_2\right)}{\partial\left(y_1, y_2\right)}$$

The magnification of images diverges when $\operatorname{det} J = 0$. The set of all image positions where this happens form closed *critical curves*, and the corresponding set of source positions define closed curves called *caustics*. Although the magnification of a point source diverges along caustic curves, that of a real-finite sized source is large but finite.

Caustics also form boundaries of regions with different numbers of images. Microlensing of a source inside the region bound by caustics always produces two more images than when the source is outside. The topology of caustics depends on the projected separation of the lens components in units of the Einstein ring radius (s) and their mass ratio (q), and is classified into three types: *close*, *resonant/intermediate*, and *wide*.

<center><img src="https://github.com/rges-pit/data-challenge-notebooks/blob/main/AAS%20Workshop/Session%20C%3A%20Binary%20Lens/Plots/caustic_structures.png?raw=1" width="1000"/></center>
<center>Fig 1: Close, resonant, and intermediate caustics for q = 0.6.</center>

For a fixed q, the boundaries of these topologies are given by,


$$\frac{q}{(1+q)^2}=\frac{\left(1-s_c\right)^3}{27 s_c^8}, \, s_w=\frac{\left(1+q^{1 / 3}\right)^{3 / 2}}{(1+q)^{1 / 2}}$$

if $s \le s_c$ (close topology), there is one diamond-shaped caustic at the center of mass of the binary lens and two triangular caustics; if $s_c \le s \le s_w$ (resonant topology), there is one large caustic at the center of mass; and if $s \ge s_w$ (wide topology) there are two diamond-shaped caustics.

<center><img src="https://github.com/rges-pit/data-challenge-notebooks/blob/main/AAS%20Workshop/Session%20C%3A%20Binary%20Lens/Plots/caustic_topology.png?raw=1" width="500"/></center>
<center>Fig 2: Boundaries of the different caustic topologies as a function of the mass ratio q of the binary lens (Gaudi, 2010).</center>

Microlensing events where the source crosses a caustic have sharp peaks in magnification on top of the smooth single lens magnification curve (as we will see in the next section) - a characteristic feature of binary lens events which can be used to easily distinguish them from single lens events.


<style>
.exercise {
    background-color: #E0E0E0;
    border-left: 8px solid #808080;
    padding: 10px 0 10px 20px;  /* top, right, bottom, left */
    margin: 20px 5px;  
    box-sizing: border-box;  
}
.exercise h2 {
    color: #808080;
    font-size: 24px;
}
.exercise p {
    margin: 0 20px;  /* Adjust this value to add space after the paragraph */
}
</style>

<div class="exercise">
    <h2>Exercise 1</h2>
    <p>Use the interactive <i>s</i> and <i>q</i> slider to plot caustics for different lens configurations. Plot all three caustic topologies and see how caustics transition from one topology to another by changing <i>s</i>. See how caustics change with <i>q</i>, and compare caustics for a planetary mass companion and a stellar mass companion. </p>
    <br>
</div>

In [ ]:

#Use this code to plot caustics with a slider for s and q
# Plot limits and styling
X_MIN, X_MAX = -1.1, 1.1
Y_MIN, Y_MAX = -1.1, 1.1


def plot_binary_caustics(log_s, log_q):
    s = 10**log_s
    q = 10**log_q

    # Masses normalized so that total binary mass = 1
    m1 = 1.0 / (1.0 + q)
    m2 = q * m1

    # Lens positions (binary lens axis along the x axis with the center of mass at the origin and the more massive lens component on the left)
    z1x = -q * s / (1.0 + q)
    z2x = s / (1.0 + q)


    # Binary lens caustic and critical curves using MulensModel
    binary_caustic = mm.CausticsBinary(s=s, q=q)
    bx, by = binary_caustic.get_caustics()
    binary_critcurves = binary_caustic.critical_curve
    bcritx, bcrity = binary_critcurves.x, binary_critcurves.y

    # Draw figure
    fig = plt.figure(figsize=(5, 5), dpi=150)
    ax = fig.add_subplot(1, 1, 1)
    plt.subplots_adjust(top=0.9, bottom=0.08, right=0.95, left=0.1, hspace=0.1, wspace=0.2)

    ax.set_xlim(X_MIN, X_MAX)
    ax.set_ylim(Y_MIN, Y_MAX)
    x_left, x_right = ax.get_xlim()
    y_low, y_high = ax.get_ylim()
    ax.set_aspect(abs((x_right - x_left) / (y_high - y_low)))

    # Plot lenses
    ax.scatter([z1x], [0.0], marker='o', s=8, color='goldenrod', label='Lens 1')
    ax.scatter([z2x], [0.0], marker='o', s=8*q, color='red', label='Lens 2')

    # Plot caustics
    ax.scatter(bx, by, marker='.', s=0.01, color='blue', alpha=0.8)
    #Plot critical curves
    ax.scatter(bcritx, bcrity, color='gray', s = 0.01, alpha=0.5, marker='.')

    #Add labels for critical curves and caustics
    gray_line = mlines.Line2D([], [], color='gray', linestyle='-', label='Critical curve')
    blue_line = mlines.Line2D([], [], color='blue', linestyle='-', label='Binary caustic')

    #Create legend
    handles, labels = ax.get_legend_handles_labels()
    handles.append(gray_line)
    handles.append(blue_line)
    labels.append(gray_line.get_label())
    labels.append(blue_line.get_label())
    legend = ax.legend(handles, labels, loc="upper left", fontsize=9)
    ax.add_artist(legend)

    ax.set_title(f's={s:.3f}, q={q:.2e}')
    plt.show()


# Sliders
log_s = FloatSlider(value=-0.5, min=-1, max=1, step=0.05, description='Log(s)')
log_q = FloatSlider(value=-3, min=-4, max=0, step=0.1, description='Log(q)')


ui = widgets.VBox([log_s, log_q])
out = widgets.interactive_output(
    plot_binary_caustics, {"log_s": log_s, "log_q": log_q}
)

display(ui, out)


## The Binary Lens Model

We need at least three additional parameters to describe a binary lens, along with the four single lens parameters ($t_0$, $u_0$, $t_E$, and $\rho$). These are:

$s$: Projected separation of the lens components in units of the Einstein ring radius $r_E$

$q$: Mass ratio between the two lens components

$\alpha$: Angle that the source trajectory makes with the binary lens axis


<center><img src="https://github.com/rges-pit/data-challenge-notebooks/blob/main/AAS%20Workshop/Session%20C%3A%20Binary%20Lens/Plots/binary_parameters.png?raw=1" width="700"/></center>
<center>Fig. 3: Binary lens event parameters</center>

The plot above shows the general set up of a binary lens microlensing event, and the conventions that we will use in the rest of this notebook. The two lenses are placed on the X-axis with the more massive lens on the left. The center of mass of the system is at the origin. $\alpha$ is the angle between the vector connecting $m_1$ to $m_2$ and the source trajectory. $\tau$ is defined the same way as the single lens case, $\tau = (t - t_0)/t_E$, and is used to parameterize the trajectory.

The Einstein ring radius is defined using the combined mass of the system: $\theta_E = \sqrt{\kappa \ (m_1 + m_2) \ \pi_{rel}}$. Unlike single lens events, the Einstein ring is not a ring around which the lensed images are produced but is instead a mathematical construct used to give relative scale to the model. In a binary lens system, images form around critical curves in the image plane, which envelope the lens masses.

Let us now see what the magnification curve for a binary lens event looks like.

In [ ]:
#We will use MulensModel to calculate the magnification curve for a given source trajectory and a binary lens model.

def get_bin_mag(params, tlist):
    """
    Calculate binary-lens magnifications using MulensModel with the VBM method.

    Parameters
    ----------
    params : dict
        Binary lens parameters. Expected keys:
            t_0   - time of closest approach (days)
            u_0   - impact parameter (Einstein radii)
            t_E   - Einstein crossing time (days)
            rho   - source radius in Einstein radii
            s     - projected lens separation (Einstein radii)
            q     - lens mass ratio (M2/M1)
            alpha - angle of source trajectory w.r.t. binary axis (degrees)
    tlist : array-like
        Times at which to evaluate the magnification (days).

    Returns
    -------
    numpy.ndarray
        Magnification values at each time in tlist.
    """
    tlist = np.array(tlist)

    model = mm.Model({
        't_0':   params['t_0'],
        'u_0':   params['u_0'],
        't_E':   params['t_E'],
        'rho':   params['rho'],
        's':     params['s'],
        'q':     params['q'],
        'alpha': params['alpha'],
    })

    # Use VBM for the entire light curve
    model.set_magnification_methods([tlist[0], 'VBM', tlist[-1]])

    magnifications = model.get_magnification(tlist)
    return magnifications


<style>
.exercise {
    background-color: #E0E0E0;
    border-left: 8px solid #808080;
    padding: 10px 0 10px 20px;  /* top, right, bottom, left */
    margin: 20px 5px;  
    box-sizing: border-box;  
}
.exercise h2 {
    color: #808080;
    font-size: 24px;
}
.exercise p {
    margin: 0 20px;  /* Adjust this value to add space after the paragraph */
}
</style>
<div class="exercise">
    <h2>Exercise 3</h2>
    <p>The cell below uses this function to plot the magnification curve, caustics and source trajectory for a given binary lens model. Vary the parameters to see how the light curve changes. See what happens when the source trajectory crosses a caustic.</p>
    <br>
</div>

In [ ]:

def plot_binary_lens_model(t_0, u_0, t_E, log_rho, log_s, log_q, alpha):
    rho = 10**log_rho
    s   = 10**log_s
    q   = 10**log_q

    params = {'t_0': t_0, 'u_0': u_0, 't_E': t_E,
              'rho': rho, 's': s, 'q': q, 'alpha': alpha}

    # Time array centered on t_0
    t = np.linspace(t_0 - 3.0*t_E, t_0 + 3.0*t_E, 3000)

    # Source trajectory using MulensModel's built-in Trajectory class
    model_traj = mm.Model({'t_0': t_0, 'u_0': u_0, 't_E': t_E,
                           's': s, 'q': q, 'alpha': alpha})
    traj = mm.Trajectory(t, parameters=model_traj.parameters)
    xs, ys = traj.x, traj.y

    # Caustics and lens positions
    binary_caustic = mm.CausticsBinary(s=s, q=q)
    bx, by = binary_caustic.get_caustics()
    z1x = -q * s / (1.0 + q)
    z2x =      s / (1.0 + q)

    # Magnification curve
    try:
        mag = get_bin_mag(params, t)
    except Exception as e:
        print(f"Magnification error: {e}")
        return

    # Dynamic view limits: cover caustics, lenses, and trajectory
    pad   = 0.3
    x_all = np.concatenate([bx, xs, [z1x, z2x]])
    y_all = np.concatenate([by, ys, [0.0, 0.0]])
    xlim  = (max(-3.0, x_all.min() - pad), min(3.0, x_all.max() + pad))
    ylim  = (max(-3.0, y_all.min() - pad), min(3.0, y_all.max() + pad))
    span  = max(xlim[1]-xlim[0], ylim[1]-ylim[0])
    xmid  = 0.5*(xlim[0]+xlim[1])
    ymid  = 0.5*(ylim[0]+ylim[1])
    xlim  = (xmid - span/2, xmid + span/2)
    ylim  = (ymid - span/2, ymid + span/2)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), dpi=130)

    # --- Left panel: lens plane ---
    ax1.scatter(bx, by, s=0.05, color='royalblue', alpha=0.9, label='Caustic')
    ax1.plot(xs, ys, color='crimson', lw=1.0, label='Source trajectory')

    # Arrowhead at trajectory midpoint showing direction of motion
    mid = len(t) // 2
    dx  = xs[mid + 1] - xs[mid - 1]
    dy  = ys[mid + 1] - ys[mid - 1]
    norm = np.hypot(dx, dy)
    half = (xlim[1] - xlim[0]) * 0.04       # arrowhead half-length ~ 4% of view width
    ax1.annotate('',
                 xy=(xs[mid] + half * dx / norm, ys[mid] + half * dy / norm),
                 xytext=(xs[mid] - half * dx / norm, ys[mid] - half * dy / norm),
                 arrowprops=dict(arrowstyle='->', color='crimson',
                                 lw=1.5, mutation_scale=14),
                 zorder=6)

    ax1.scatter([z1x], [0.0], marker='*', s=120, color='goldenrod',
                zorder=7, label=r'$m_1$')
    ax1.scatter([z2x], [0.0], marker='*', s=max(20, 120*q),
                color='darkorange', zorder=7, label=r'$m_2$')
    ax1.set_xlim(*xlim)
    ax1.set_ylim(*ylim)
    ax1.set_aspect('equal')
    ax1.set_xlabel(r'$x\;(\theta_E)$')
    ax1.set_ylabel(r'$y\;(\theta_E)$')
    ax1.set_title(fr'$s={s:.3f},\; q={q:.2e},\; \alpha={alpha:.1f}°$')
    ax1.legend(fontsize=7, loc='upper right', markerscale=1.5)

    # --- Right panel: light curve ---
    ax2.plot(t, mag, color='crimson', lw=0.9)
    ax2.set_xlabel(r'$t$ (days)')
    ax2.set_ylabel('Magnification')
    ax2.set_title(fr'$t_0={t_0:.1f},\; u_0={u_0:.3f},\; t_E={t_E:.1f},\; \rho={rho:.2e}$')
    

    plt.tight_layout()
    plt.show()


# ---- Sliders ----
slider_layout = widgets.Layout(width='420px')
slider_style  = {'description_width': '110px'}

w_t0    = FloatSlider(value=0.0,             min=-50.0, max=50.0,  step=0.5,
                      description='t₀ (days)',  style=slider_style, layout=slider_layout)
w_u0    = FloatSlider(value=-0.53,           min=-2.0,  max=2.0,   step=0.01,
                      description='u₀',          style=slider_style, layout=slider_layout)
w_tE    = FloatSlider(value=30.0,            min=1.0,   max=200.0, step=1.0,
                      description='tE (days)',   style=slider_style, layout=slider_layout)
w_lrho  = FloatSlider(value=-3.0,            min=-4.0,  max=-0.5,  step=0.1,
                      description='log₁₀(ρ)',    style=slider_style, layout=slider_layout)
w_ls    = FloatSlider(value=np.log10(1.4),   min=-0.6,  max=0.8,   step=0.02,
                      description='log₁₀(s)',    style=slider_style, layout=slider_layout)
w_lq    = FloatSlider(value=-2.0,            min=-4.0,  max=0.0,   step=0.1,
                      description='log₁₀(q)',    style=slider_style, layout=slider_layout)
w_alpha = FloatSlider(value=30.0,            min=0.0,   max=360.0, step=1.0,
                      description='α (deg)',     style=slider_style, layout=slider_layout)

ui = widgets.VBox([w_t0, w_u0, w_tE, w_lrho, w_ls, w_lq, w_alpha])
out = widgets.interactive_output(
    plot_binary_lens_model,
    {'t_0': w_t0, 'u_0': w_u0, 't_E': w_tE, 'log_rho': w_lrho,
     'log_s': w_ls, 'log_q': w_lq, 'alpha': w_alpha}
)

display(out, ui)

### Finite source effects

If the magnification changes rapidly over a distance comparable to the source size, then finite source effects become important. This effect is parameterized by $\rho = r_{\star}/\theta_{\rm E}$, where $r_{\star}$ is the angular radius of the source. Finite source effects are often seen in binary lens events since caustic crossings or caustic approaches can resolve the source size. In the presence of finite source effects, the amplitude of light curve peaks produced by caustic crossings decreases and they become more rounded. 

<style>
.exercise {
    background-color: #E0E0E0;
    border-left: 8px solid #808080;
    padding: 10px 0 10px 20px;  /* top, right, bottom, left */
    margin: 20px 5px;  
    box-sizing: border-box;  
}
.exercise h2 {
    color: #808080;
    font-size: 24px;
}
.exercise p{
    margin: 0 20px;  /* Adjust this value to add space after the paragraph */
}
</style>
<div class="exercise">
    <h2>Exercise 4</h2>
    <p>Use the sliders in the cell above to create a caustic crossing event. 
    See how the light curve changes when the you vary the source size parameter.</p> 
    <br>
</div>

## Fitting a Binary Lens Model

Now that we have the tools to create a binary lens model, let us get our hands dirty with real microlensing data and to try to fit a binary lens model to it! The cell below loads a simulated Roman light curve from the 2017 Roman Data Challenge. The data has been cropped to only include data points from a small window around the event peak (a little less than one Roman season) to ensure computations are tractable in the time we have during this session. We will create a MulensModel Data object to store the light curve data. 

In [ ]:
%matplotlib widget

In [ ]:
data_Roman_W149 = mm.MulensData(file_name="dc1_186_W149.txt", #data file must have columns Time, Mag/Flux, Err in this order
                                phot_fmt='mag',
                                chi2_fmt = 'flux',
                                ephemerides_file=None, #ephemerides file is used to specify the ephemerides of the telescope to calculate parallax effects. We will not fit for parallax in this session.
                                plot_properties={'color': '#a859e4',
                                                 'fmt': 'o',
                                                 'markersize': 1,
                                                 'show_errorbars': True,
                                                 'label': 'Roman W149'
                                                 },
                                bandpass='H' #Primarily used to calculate limb darkening of the source star.
                               )


#Plot the data
fig, ax = plt.subplots(figsize=(8, 6))
data_Roman_W149.plot(phot_fmt='mag', subtract_2450000=True, label='Roman W149')
plt.xlabel('HJD - 2450000')
plt.ylabel('Magnitude')
plt.legend()


### <font face="Helvetica" size="5"> Single lens fit </font>

This event has an obvious perturbation on top of the smooth single lens light curve, indicating that it is a binary lens event. Let us first fit a single lens model to determine the single lens parameters. First we need to find a good guess for the single lens parameters ($t_0$, $u_0$, $t_E$)

<style>
.exercise {
    background-color: #E0E0E0;
    border-left: 8px solid #808080;
    padding: 10px 0 10px 20px;  /* top, right, bottom, left */
    margin: 20px 5px;  
    box-sizing: border-box;  
}
.exercise h2 {
    color: #808080;
    font-size: 24px;
}
.exercise p{
    margin: 0 20px;  /* Adjust this value to add space after the paragraph */
}
</style>
<div class="exercise">
    <h2>Exercise 5</h2>
    <p>Find an initial guess for the single lens parameters through qualitative analysis of the light curve. Use the sliders below to find the values of the light curve at different points.</p> 
    <br>
</div>

In [ ]:
from ipywidgets import interact, FloatSlider

# Define the interactive plot function
def plot_with_lines(vline_pos, hline_pos):
    fig = plt.figure(figsize=(10, 6))
    plt.errorbar(data_Roman_W149.time, data_Roman_W149.mag, yerr=data_Roman_W149.err_mag, marker='o', color='black', ls='', markersize=1)
    plt.gca().invert_yaxis()
    plt.xlabel('HJD')
    plt.ylabel('Mag')
    
    ## Vertical and horizontal lines controlled by sliders
    plt.axvline(x=vline_pos, color='red', linestyle='--', linewidth=2, label=f'vline={vline_pos:.2f}')
    plt.axhline(y=hline_pos, color='blue', linestyle='--', linewidth=2, label=f'hline={hline_pos:.2f}')
    plt.legend()
    plt.tight_layout()
    plt.show()

# Create sliders with appropriate ranges based on the data
x_min, x_max = data_Roman_W149.time.min(), data_Roman_W149.time.max()
y_min, y_max = data_Roman_W149.mag.min(), data_Roman_W149.mag.max()  # Note: inverted because y-axis is inverted

# Create interactive plot with sliders
interact(plot_with_lines,
         vline_pos=FloatSlider(value=data_Roman_W149.time.min(), min=x_min, max=x_max, step=0.1, description='Vertical Line:', style={'description_width': 'initial'}),
         hline_pos=FloatSlider(value=data_Roman_W149.mag.min(), min=y_min, max=y_max, step=0.01, description='Horizontal Line:', style={'description_width': 'initial'}));

In [ ]:
#Do your computations here:
# Time of the peak of the event
t_0i = 2458394.40

# Baseline magnitude
mag_0 = 19.61

# Change in magnitude
delta_mag_max = (19.61 - 18.67)

# Maximum magnification
A_max = 10**(delta_mag_max/2.5)

# Impact parameter
u_0i = np.sqrt(2.0*(-1 + A_max/np.sqrt(A_max**2 - 1)))

#To find t_E:
#A(t_E) = (u_0^2 + 3)/(sqrt(u_0^2 + 1)sqrt(u_0^2 + 5)

A_tE = (u_0i**2 + 3)/(np.sqrt(u_0i**2 + 1)*np.sqrt(u_0i**2 + 5))
print("Magnification at t_E: ", A_tE)

# Change in magnitude 
delta_mag = 2.5*np.log10(A_tE)
print("Magnitude at t_E: ", mag_0 - delta_mag)

# Time at t_0 - t_E
t = 2458389.90

# Einstein timescale
t_Ei = np.abs(t_0i - t)

print(t_0i, u_0i, t_Ei)

**Your Guess**:

t_0 = 2458394.4 HJD

u_0 = 0.45 

t_E = 4.5 days


Let us now create MulensModel model and event objects as before. We will flag the planetary perturbation as bad data to fit only the main event due to the star. This is possible for the light curve we have since the event looks mostly like a single lens event with a small planet perturbation and cannot be done if you have a resonant caustic crossing or a binary star event with very large deviations.

In [ ]:
#Flag planet perturbation as bad data
t_planet_start = 2458398.5
t_planet_end = 2458401.5
# Flag data related to the planet
flag_planet = (data_Roman_W149.time > t_planet_start) & (data_Roman_W149.time < t_planet_end) | np.isnan(data_Roman_W149.err_mag)
# Exclude those data from the fitting (for now)
data_Roman_W149.bad = flag_planet

#Create a PSPL model
params_pspl = {'t_0': t_0i, 'u_0': u_0i, 't_E': t_Ei}
model_pspl = mm.Model(params_pspl, coords=None, ephemerides_file=None) #Coords and ephemerides file are required for parallax calculations
model_pspl.set_magnification_methods([data_Roman_W149.time.min(), 'point_source', data_Roman_W149.time.max()])

#Create an event object
event_roman = mm.Event(datasets=[data_Roman_W149], model=model_pspl)

init_chi2 = event_roman.get_chi2()
FS, FB = event_roman.get_ref_fluxes()

#Plot the data and initial model
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8), gridspec_kw={'height_ratios': [3, 1]},  sharex=True)
plt.sca(ax1)
data_Roman_W149.plot(phot_fmt='flux', subtract_2450000=True, label=fr'$\chi^2 = {init_chi2:.2f}$')
model_mag = model_pspl.get_magnification(data_Roman_W149.time)
model_flux = FS * model_mag + FB
plt.plot(data_Roman_W149.time - 2450000, model_flux, color='black', label='Initial PSPL model')
#Plot the residuals
plt.sca(ax2)
plt.errorbar(data_Roman_W149.time - 2450000, data_Roman_W149.flux - model_flux, yerr=data_Roman_W149.err_flux, color='#a859e4', label='Residuals')
plt.axhline(0, color='black', linestyle='-')
ax1.set_ylabel('Flux')
ax2.set_xlabel('HJD - 2450000')
ax2.set_ylabel('Residuals')
ax1.legend()
fig.tight_layout()
plt.show()

#You can also use a built-in plot function:
#event_pspl.plot(t_range=[data_Roman_W149.time.min(), data_Roman_W149.time.max()], residuals=True, show_errorbars=True, legend=True, subtract_2450000=True)
#But it gives you lesser freedom and it plots in magnitude space which I don't prefer.

A flexible fitting function to perform basic downhill simplex minimization using scipy's Nelder-Mead implementation. Can be used to fit binary and single lenses. Accepts bounds for parameters and allows you to fix certain parameters and fit only the rest if you wish so. You can also fit t_E, s, q, rho in log space by adding the "log_" prefix to these parameter names in the params_to_fit list

In [ ]:
def chi2(x0, params_to_fit, event):
    """
    Calculates chi-squared by updating the event model with the given parameters.
    
    Parameters
    ----------
    x0 : array-like
        Parameter values (in the order of params_to_fit)
    params_to_fit : list[str]
        Parameter names. Use 'log_' prefix for t_E, s, q, rho to fit in log space.
    event : mm.Event
        MulensModel Event object
        
    Returns
    -------
    float
        Chi-squared value
    """
    log_param_map = {'log_t_E': 't_E', 'log_s': 's', 'log_q': 'q', 'log_rho': 'rho'}
    
    for i, param_name in enumerate(params_to_fit):
        if param_name in log_param_map:
            actual_param = log_param_map[param_name]
            setattr(event.model.parameters, actual_param, 10**x0[i])
        else:
            setattr(event.model.parameters, param_name, x0[i])
    
    event.fit_fluxes()
    return event.get_chi2()


def simple_fit(event: mm.Event, params_to_fit: list[str], bounds: dict | None = None, maxfev: int = 1000):
    """
    Fit a single or binary lens model using downhill simplex method.

    Parameters
    ----------
    event : mm.Event (Required)
        MulensModel Event object with the initial model set.
    params_to_fit : list[str] (Required)
        Parameter names to fit. Use 'log_' prefix (log_t_E, log_s, log_q, log_rho)
        to fit in log space. Parameters NOT in this list are held fixed.
    bounds : dict | None, optional
        Map of parameter name -> (min, max). If None, use defaults.
    maxfev : int, optional
        Maximum number of function evaluations. Default is 1000.

    Returns
    -------
    fin_params : dict
        Dictionary with final parameters (in linear space)
    fin_chi2 : float
        Final chi-squared value
    reduced_chi2 : float
        Reduced chi-squared = chi2/dof
    """
    t = event.datasets[0].time
    n_data = len(event.datasets[0].flux) - sum(event.datasets[0].bad)
    
    # Map log_* names to their actual model parameter names
    log_param_map = {'log_t_E': 't_E', 'log_s': 's', 'log_q': 'q', 'log_rho': 'rho'}
    
    # Default bounds (in fitting space, i.e., log for log_* params)
    default_bounds = {
        't_0': (t.min(), t.max()),
        't_E': (0.1, t.max() - t.min()),
        'u_0': (-10.0, 10.0),
        'rho': (1e-5, 1.0),
        's': (1e-2, 20.0),
        'q': (1e-7, 1.0),
        'alpha': (0.0, 360.0),
        'log_s': (-2, 2),
        'log_q': (-7, 0),
        'log_rho': (-5, 0),
        'log_t_E': (-1, 3),
    }
    
    # Merge user bounds with defaults
    bounds_map = default_bounds.copy()
    if bounds is not None:
        bounds_map.update(bounds)
    
    # Build initial parameter vector from event model
    x0 = []
    for param_name in params_to_fit:
        if param_name in log_param_map:
            actual_param = log_param_map[param_name]
            val = getattr(event.model.parameters, actual_param)
            x0.append(np.log10(val))
        else:
            x0.append(getattr(event.model.parameters, param_name))
    x0 = np.array(x0)
    
    # Build bounds list for optimizer
    bounds_list = [bounds_map.get(p, (None, None)) for p in params_to_fit]
    
    # Run optimization
    result = op.minimize(
        chi2, x0, args=(params_to_fit, event),
        bounds=bounds_list,
        method='Nelder-Mead',
        options={'xatol': 1e-8, 'fatol': 1e-8, 'adaptive': False, 'maxfev': maxfev}
    )
    
    # Extract final chi2
    fin_chi2 = float(result.fun) if not isinstance(result.fun, np.ndarray) else float(result.fun.flat[0])
    
    # Build final parameter dictionary (convert log params back to linear)
    fin_params = {}
    for i, param_name in enumerate(params_to_fit):
        if param_name in log_param_map:
            actual_param = log_param_map[param_name]
            fin_params[actual_param] = 10**result.x[i]
        else:
            fin_params[param_name] = result.x[i]
    
    # Add fixed parameters (those in the model but not being fit)
    all_model_params = dict(event.model.parameters.parameters)
    for key, val in all_model_params.items():
        if key not in fin_params:
            fin_params[key] = val
    
    # Degrees of freedom: n_data - n_fitted_params - 2 (for Fs, Fb)
    dof = n_data - len(params_to_fit) - 2
    delta_chi2 = fin_chi2 - dof
    
    return fin_params, fin_chi2, delta_chi2

Let us use this function to first fit a single lens model to the light curve.

In [ ]:
fin_params_pspl, fin_chi2, delta_chi2 = simple_fit(event_roman, params_to_fit=['t_0', 'log_t_E', 'u_0'])
#Plot data and final model
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8), gridspec_kw={'height_ratios': [3, 1]},  sharex=True)
plt.sca(ax1)
data_Roman_W149.plot(phot_fmt='flux', subtract_2450000=True, label=fr'$\chi^2 = {fin_chi2:.2f}$')
model_mag = event_roman.model.get_magnification(data_Roman_W149.time)
FS, FB = event_roman.get_ref_fluxes()
model_flux = FS * model_mag + FB
plt.plot(data_Roman_W149.time - 2450000, model_flux, color='black', label=rf'Final PSPL model, $\Delta \chi^2 = {delta_chi2:.2f}$')
#Plot the residuals
plt.sca(ax2)
plt.errorbar(data_Roman_W149.time - 2450000, data_Roman_W149.flux - model_flux, yerr=data_Roman_W149.err_flux, color='#a859e4', label='Residuals')
plt.axhline(0, color='black', linestyle='-')
ax1.set_ylabel('Flux')
ax2.set_xlabel('HJD - 2450000')
ax2.set_ylabel('Residuals')
ax1.legend()
fig.tight_layout()
plt.show()

print("Final single lens parameters: ", fin_params_pspl, "FS: ", FS, "FB: ", FB)


Great! The final model looks like a fairly good fit to the light curve minus the perturbation due to the lens companion. Now let us try to fit a binary lens model using the same function, from random initial guesses for the additional binary parameters. First let us unflag the planet perturbation. We will also use a combination of point lens point source magnification methods when we are not close to the planet perturbation and binary lens magnification calculations for the planet perturbation itself. This will help speed up calculations.

### <font face="Helvetica" size="5"> Binary lens fit from a random initial guess </font>

In [ ]:
#unflag the planet perturbation
event_roman.datasets[0].bad = np.isnan(event_roman.datasets[0].err_mag)

#Start from best fit single lens parameters
t0i = fin_params_pspl['t_0']
tEi = fin_params_pspl['t_E']
u0i = fin_params_pspl['u_0']
#And random guesses for the binary parameters
logsi = np.random.uniform(-1, 1)
logqi = np.random.uniform(-5, 0)
alphai = np.random.uniform(0.0, 360.0)
rhoi = 10**np.random.uniform(-5, 0)

params_binary = {
    't_0': t0i,
    't_E': tEi,
    'u_0': u0i,
    'rho': rhoi,
    's': 10**logsi,
    'q': 10**logqi,
    'alpha': alphai
}

#Update the event object with the new model
event_roman.model = mm.Model(params_binary, coords=None, ephemerides_file=None)
#We will use the point-source-point-lens magnification method everywhere except for the planet perturbation and VBM for the planet perturbation
event_roman.model.set_magnification_methods([data_Roman_W149.time.min(), 'point_source_point_lens', t_planet_start, 'VBM', t_planet_end, 'point_source_point_lens', data_Roman_W149.time.max()])

#Fit using the simple_fit function
fin_params, fin_chi2_bin, delta_chi2_bin = simple_fit(event_roman, params_to_fit=['t_0', 't_E', 'u_0', 'log_rho', 'log_s', 'log_q', 'alpha'], maxfev=500)
FS, FB = event_roman.get_ref_fluxes()


#plot the data and the model
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8), gridspec_kw={'height_ratios': [3, 1]},  sharex=True)
plt.sca(ax1)
data_Roman_W149.plot(phot_fmt='flux', subtract_2450000=True, label=fr'$\chi^2 = {fin_chi2_bin:.2f}$')
model_mag = event_roman.model.get_magnification(data_Roman_W149.time)
FS, FB = event_roman.get_ref_fluxes()
model_flux = FS * model_mag + FB
delta_chi2 = fin_chi2_bin - fin_chi2
plt.plot(data_Roman_W149.time - 2450000, model_flux, color='black', label=rf'Final binary model, $\Delta \chi^2 = {delta_chi2_bin:.2f}$')
#Plot the residuals
plt.sca(ax2)
plt.errorbar(data_Roman_W149.time - 2450000, data_Roman_W149.flux - model_flux, yerr=data_Roman_W149.err_flux, color='#a859e4', label='Residuals')
plt.axhline(0, color='black', linestyle='-')
ax1.set_ylabel('Flux')
ax2.set_xlabel('HJD - 2450000')
ax2.set_ylabel('Residuals')
ax1.legend()
fig.tight_layout()
plt.show()

print("Final binary lens parameters: ", fin_params, "FS: ", FS, "FB: ", FB)


This was a disaster! A binary lens microlensing model is too complex to be fit by a downhill simplex algorithm. The likelihood (or $\chi^2$) space of a binary lens event is very rugged and riddled with deep local minima. So any gradient descent-like method will fail or get stuck in a local minimum unless we are very close to the true solution.

But we don't have to start from a completely random guess for the binary lens parameters. The shape and poistion of the planetary/stellar companion perturbation can be used to get a rough estimate of the binary parameters following the procedure outlined in [Gaudi & Gould (1997)](https://ui.adsabs.harvard.edu/abs/1997ApJ...486..675G). 

Let us try to get a better guess for the binary lens parameters:

### <font face="Helvetica" size="5"> Where is the planet? </font>

In [ ]:
interact(plot_with_lines,
         vline_pos=FloatSlider(value=t0i, min=x_min, max=x_max, step=0.1, description='Vertical Line:', style={'description_width': 'initial'}),
         hline_pos=FloatSlider(value=data_Roman_W149.flux.min(), min=y_min, max=y_max, step=0.01, description='Horizontal Line:', style={'description_width': 'initial'}));

First we have to find t_planet, the time when the planetary anomaly occurs. Use the plot above to determine that.

In [ ]:
# Time of the planet perturbation
t_planet = 2458400.30

Now, based on the information gained above we can find the position of the images.

In [ ]:
# Time scaled to the Einsein timescale
tau = np.abs(t_planet - fin_params_pspl['t_0']) / fin_params_pspl['t_E']

# source-lens separation
u = np.sqrt(fin_params_pspl['u_0']**2 + tau**2)

# Position of the images
y_plus = 0.5 * np.sqrt(u**2 + 4.) + u
y_minus = 0.5 * np.sqrt(u**2 + 4.) - u

print(y_plus, y_minus)

### <font face="Helvetica" size="5"> Find planet-star separation and the angle of the source trajectory </font>

To find the planet-star separation, we have to determine whether minor or major image is perturbed.
- Minor image perturbation occurs when there is a dip in the light curve during a planetary anomaly.

If the minor image is perturbed: $s = y_-$.

<center><img src="https://github.com/rges-pit/minicourses/blob/main//chapter4/images/Yee_sagan_minor_pert.png?raw=true" width="500"/></center>
<center>Minor image perturbation light curve, plot from Jennifer Yee's presentation during the Sagan Exoplanet Workshop 2017 for this exercise.</center>
- Major image perturbation occurs when there is a bump in the light curve during a planetary anomaly.

If the major image is perturbed: $s = y_+$

<center><img src="https://github.com/rges-pit/minicourses/blob/main/chapter4/images/Yee_sagan_major_pert.png?raw=true" width="500"/></center>
<center>Major image perturbation light curve, plot from Jennifer Yee's presentation during the Sagan Exoplanet Workshop 2017 for this exercise.</center>


⚠️ If your bump has a little dip that doesn't go back to the level of the main event, it would still be a major image perturbation rather than a minor image perturbation. The dip between two peaks can occur when the source is travelling inside the major image caustic.

In [ ]:
#Find s 
si =  y_minus

Now we can find the value of the angle of the source trajectory, alpha, using values we determined earlier. Due to different geometric conventions, the correct value might be $\frac{\pi}{2}$ or $\pi$ away from the value you calculated.

In [ ]:
# angle between trajectory and binary axis
alphai = np.rad2deg(np.arctan2(fin_params_pspl['u_0'], tau))
print("Angle of the source trajectory: ", alphai)

### <font face="Helvetica" size="5"> Find  source radius and planet-star mass ratio </font>

Finding the source size and planet-star mass ratio through our qualitative method is the hardest to do. The source size $\rho$ can be easily constrained if there is a caustic crossing. 

If the source size is less than the size of the caustic, then you will see distinct peaks for the caustic entry and caustic exit. In this case, $$\rho = \theta_{\star}/\theta_E = t_{\rm caustic}/t_E$$, where $t_{\rm caustic}$ is the duration of the caustic entry or exit. 

If the source size is comparable to or larger than the caustic, then you will see only one rounded bump, and $\rho$ can be estimated similarly from the duration of the bump (this is only accurate if the source is much larger than the caustic). 

To estimate $q$, you can use one of two approximate relations depending on the source size. If $\rho$ is smaller than the caustic, then $q$ is just the ratio of the timescale of the planet perturbation squared ($t_{\rm E,p}^2$) to $t_E^2$ since $t_E^2 \propto M_{\star}$ and $t_{E,p}^2 \propto M_p$. Here $t_{\rm E,p}$ is half the time duration of the entire planetary perturbation (since it is the time taken to cross the planetary Einstein ring).

If $\rho$ is larger than the caustic and you see one blended peak, then the amplitude of the bump is related to the mass ratio, 

$A_p = \frac{2q}{\rho^2}$    ([**Gould & Gaucherel, 1997**](https://ui.adsabs.harvard.edu/abs/1997ApJ...477..580G/abstract))

where $A_p$ is the amplitude of the planetary perturbation. 

In our example, there is no caustic crossing, and the dip is due to a ridge of demagnification between the two triangular caustics in the close topology. This makes it hard to constrain the source size very well, but we can say that it is smaller than the distance between the two caustics. Let us first calculate $q$ from the timescale of the planetary perturbation. 

 <style>
.exercise {
    background-color: #E0E0E0;
    border-left: 8px solid #808080;
    padding: 10px 0 10px 20px;  /* top, right, bottom, left */
    margin: 20px 5px;  
    box-sizing: border-box;  
}
.exercise h2 {
    color: #808080;
    font-size: 24px;
}
.exercise p{
    margin: 0 20px;  /* Adjust this value to add space after the paragraph */
}
</style>
<div class="exercise">
    <h2>Exercise 6</h2>
    <p>Find an initial guess for the planet-star mass ratio q using the method outlined above. Use the sliders created previously in the notebook to find the timescale of the event. </p> 
    <br>
</div>

In [ ]:
#Find q
t_Ep = (2458400.70 - 2458399.80)/2 
qi = t_Ep**2/fin_params_pspl['t_E']**2
print("Planet-star mass ratio: ", qi)

We can get a rough estimate of the source size even though there is no caustic crossing, since we know that it is smaller than the separation between the two planetary caustics. The separation between the planetary caustics is roughly $4q^{1/2}(1-s^2)^{1/2}/s$ ([**Han, 2006**](https://ui.adsabs.harvard.edu/abs/2006ApJ...638.1080H/abstract)). Estimate $\rho$ by assuming that it is $\sim 10 \%$ of this width.

In [ ]:
#Find rho
rhoi = 0.1 * 4*qi**0.5*(1-si**2)**0.5/si
print("Source size: ", rhoi)

Great! Now we have a better (hopefully) guess for the planetary parameters. Let us try to fit the binary model starting from these parameters. We will use Markov Chain Monte Carlo to do the minimization. This is a better method for binary lens models since the stochastic nature of this algorithm allows it to escape local minima where a gradient descent or downhill simplex algorithm would get stuck.

In the cells below we will implement the MCMC algorithm to fit a binary lens model and find posterior distributions for all parameters. We have defined a posterior probability function by using the $\chi^2$ likelihood for a model, and flat priors as long as the parameters are within user-defined bounds. We will try to apply fairly strict bounds for all parameters based on what we think we know about the planet perturbation. Some parameters like $\rho$ are not very well constrained by our analytic method. We will also not apply strict bounds on $\alpha$.

The [emcee](https://emcee.readthedocs.io/en/stable/) package is used to run the MCMC algorithm. It is called by a custom MCMC class for microlensing modelling that calculates various convergence staistics, determines burn-in period, finds the MLE solution and confidence intervals, and plots a corner plot of posteriors as well as chains for all walkers. The MCMC class can use multiple cores/threads to parallelize the computation if you are running this on a machine with multiple cores (does not work on Google colab).

In [ ]:
# Define the log prior function - used to set bounds on parameters
def log_prior(x0, params_to_fit, event, bounds: dict | None = None):
    """
    Uniform prior with bounds on parameters.

    Parameters
    ----------
    x0 : np.ndarray
        Parameter vector (values for params_to_fit)
    params_to_fit : list[str]
        Parameter names. Use 'log_' prefix for t_E, s, q, rho to fit in log space.
    event : mm.Event
        MulensModel Event object (used to get time range for default bounds)
    bounds : dict | None, optional
        Map of parameter name -> (min, max). If None, use defaults.
    
    Returns
    -------
    float
        0 if within bounds, -np.inf otherwise
    """
    t = event.datasets[0].time
    
    default_bounds = {
        't_0': (t.min(), t.max()),
        't_E': (0.1, t.max() - t.min()),
        'u_0': (-10.0, 10.0),
        'rho': (1e-5, 1.0),
        's': (1e-2, 20.0),
        'q': (1e-7, 1.0),
        'alpha': (0.0, 360.0),
        'log_t_E': (-1, 3),
        'log_s': (-2, 2),
        'log_q': (-7, 0),
        'log_rho': (-5, 0),
    }
    bounds_map = default_bounds.copy()
    if bounds is not None:
        bounds_map.update(bounds)

    for i, key in enumerate(params_to_fit):
        lower, upper = bounds_map.get(key, (-np.inf, np.inf))
        if x0[i] < lower or x0[i] > upper:
            return -np.inf
    return 0.0


def log_likelihood(x0, params_to_fit, event):
    """
    Log likelihood using MulensModel Event chi-squared.

    Parameters
    ----------
    x0 : np.ndarray
        Parameter vector
    params_to_fit : list[str]
        Parameter names
    event : mm.Event
        MulensModel Event object
    
    Returns
    -------
    float
        -0.5 * chi2
    """
    chi2_val = chi2(x0, params_to_fit, event)
    return -0.5 * chi2_val


def log_probability(x0, params_to_fit, event, bounds: dict | None = None):
    """
    Log probability = log prior + log likelihood.

    Parameters
    ----------
    x0 : np.ndarray
        Parameter vector
    params_to_fit : list[str]
        Parameter names
    event : mm.Event
        MulensModel Event object
    bounds : dict | None, optional
        Parameter bounds
    
    Returns
    -------
    float
        Log probability
    """
    lp = log_prior(x0, params_to_fit, event, bounds=bounds)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(x0, params_to_fit, event)



In [ ]:
#Define and plot initial model

from microlens_emcee import micro_mc

#flip the sign of u0i
u0i = -u0i
params_to_fit = ['t_0', 't_E', 'u_0', 'log_rho', 'log_s', 'log_q', 'alpha']
#Update model parameters in the event object
params_binary = {
    't_0': t0i,
    't_E': tEi,
    'u_0': u0i,
    'rho': rhoi,
    's': si,
    'q': qi,
    'alpha': alphai
}

event_roman.model = mm.Model(params_binary, coords=None, ephemerides_file=None)
event_roman.model.set_magnification_methods([data_Roman_W149.time.min(), 'point_source_point_lens', t_planet_start, 'VBM', t_planet_end, 'point_source_point_lens', data_Roman_W149.time.max()])

#Plot data and initial model
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8), gridspec_kw={'height_ratios': [3, 1]},  sharex=True)
plt.sca(ax1)
data_Roman_W149.plot(phot_fmt='flux', subtract_2450000=True, label='W149')
model_mag = event_roman.model.get_magnification(data_Roman_W149.time)
FS, FB = event_roman.get_ref_fluxes()
model_flux = FS * model_mag + FB
delta_chi2 = event_roman.get_chi2() - len(data_Roman_W149.time) + 9
plt.plot(data_Roman_W149.time - 2450000, model_flux, color='black', label=rf'Initial binary model, $\Delta \chi^2 = {delta_chi2:.2f}$')
plt.title(rf'$t_0 = {t0i:.2f}$, $t_E = {tEi:.2f}$, $u_0 = {u0i:.3f}$, $\rho = {rhoi:.2e}$, $s = {si:.2f}$, $q = {qi:.2e}$, $\alpha = {alphai:.1f}$')
#Plot the residuals
plt.sca(ax2)
plt.errorbar(data_Roman_W149.time - 2450000, data_Roman_W149.flux - model_flux, yerr=data_Roman_W149.err_flux, color='#a859e4', label='Residuals')
plt.axhline(0, color='black', linestyle='-')
ax1.set_ylabel('Flux')
ax2.set_xlabel('HJD - 2450000')
ax2.set_ylabel('Residuals')
ax1.legend()

In [ ]:
#Fit with MCMC

#Apply more stringent bounds on single lens parameters since we have a fairly good guess of what they are
informed_guess_mcmc = micro_mc(event_roman, params_to_fit, log_probability, bounds={'t_0': (t0i - tEi, t0i + tEi), 't_E': (0, tEi + 20), 'u_0': (u0i - 0.1, u0i + 0.1), 'log_rho': (-5, -0.5), 'log_s': (-1, 0), 'log_q': (-4, -1)})

#Run MCMC analysis with all plots enabled
results = informed_guess_mcmc.perform_mcmc_analysis(steps=500, walkers=25, param_scales=np.array([1e-1, 1e-1, 1e-2, 1e-1, 1e-2, 1e-2, 1e-1]),
                               verbose=True, plot_corner=True, plot_fit=True,
                               plot_traces=True, plot_convergence=False, n_threads=1,
                               pool_init_data={
                                   'data_file': 'dc1_186_W149.txt',
                                   'data_kwargs': {'phot_fmt': 'mag', 'chi2_fmt': 'flux', 'ephemerides_file': None},
                                   'model_params': params_binary,
                                   'mag_methods': [data_Roman_W149.time.min(), 'point_source_point_lens', 2458398.0, 'VBM', 2458401.0, 'point_source_point_lens', data_Roman_W149.time.max()],
                               })
#save_chains='informed_guess_mcmc_2'

We need to run the MCMC for much longer (more steps and walkers) or start much closer to the true answer to get a better converged solution. Since running a full MCMC will take a long time, we have stored an MCMC run (full chains and posterioirs) that you can just load below and plot

In [ ]:
import types, json
import numpy as np
from microlens_emcee import micro_mc

# ── Load saved chains, posteriors, and metadata ───────────────────────────────
prefix     = 'informed_guess_mcmc'
chains     = np.load(f'{prefix}_chains.npy')      # shape: (n_walkers, n_steps, n_params)
posteriors = np.load(f'{prefix}_posteriors.npy')   # shape: (n_samples, n_params)
lnprob = np.load(f'{prefix}_lnprob.npy')

with open(f'{prefix}_metadata.json') as f:
    meta = json.load(f)

params_to_fit  = meta['params_to_fit']
burn_in        = meta['burn_in']
mle_delta_chi2 = meta['mle_delta_chi2']
mle_linear     = meta['mle_linear']

# ── Reconstruct a micro_mc instance for plotting ──────────────────────────────
# event_roman and log_probability are defined in earlier cells
loaded_mcmc = micro_mc(event_roman, params_to_fit, log_probability)

# Apply MLE parameters (linear space) to the event model before plotting
for param, value in mle_linear.items():
    setattr(event_roman.model.parameters, param, value)

# ── Trace plots ───────────────────────────────────────────────────────────────
# plot_mcmc_traces only needs sampler.chain, so we wrap the loaded array
mock_sampler = types.SimpleNamespace(chain=chains, lnprobability=lnprob) #\mock_sampler = types.SimpleNamespace(chain=chains, lnprobability=lnprob)
micro_mc.plot_mcmc_traces(mock_sampler, params_to_fit, burn_in)

# ── Data + best-fit model ─────────────────────────────────────────────────────
loaded_mcmc.plot_mcmc_fit(mle_delta_chi2=mle_delta_chi2)

# ── Corner plot ───────────────────────────────────────────────────────────────
loaded_mcmc.plot_corner_mcmc(samples=posteriors, param_names=params_to_fit)

The MLE solution obtained by the MCMC run looks pretty good! This took 2 $\sim 2$ hour runs with 9 parallel threads (with the second run starting from the MLE solution obtained by the first). We can see from the posteriors that $\rho$ is not well constrained at all, as we expected.

While we were able to get a fairly good fit to the light curve, we had to apply very stringent bounds on the parameters to perform this fit and therefore be quite confident about our initial guess. Furthermore, microlensing events often have several degenerate solutions, a few of which are mathematical and cannot be resolved with the light curve alone (references). This method does not allow us to explore these degenerate solutions. The procedure used to get an initial guess also breaks down for light curves with resonant caustic perturbations and binary star events (see below for an example). 

There are more robust methods that people have come up with to search the vast and complex parameter space of microlensing. Two of the most popular methods are the Initial Condition Grid Search (reference?) and Template Matching ([Bozza et al 2024](https://ui.adsabs.harvard.edu/abs/2024A%26A...688A..83B/abstract)). These methods are the ones currently being used in Roman's event modelling pipeline. We will demonstrate the first method below.

<center><img src="https://github.com/rges-pit/sagan_session1/blob/binary-lens-edits/Binary-Lens-Fitting/dc1_071_binlens.png?raw=1" width="1000"/></center>
<center>Example light curve of a binary star lens microlensing event.</center>

## Initial Condition Grid Search

The idea of a grid search is to search for the best starting guesses on a large grid of binary parameters ($s$, $q$, and $\alpha$). The best *n* solutions are picked from the results of this search and a global minimum search is conducted using MCMC starting from these parameters. This is the most robust method to ensure that we find the global minimum, although it is also very computationally expensive. The other advantage of this method is its ability to identify degenerate solutions. 

There are many different prescriptions for a grid search that researchers have used in the past. They differ in the parameters used to create the grid, the minimization routines used, the method used to calculate magnifications, and the procedure followed to identify the global minimum once the grid search is completed. In the demonstration of a grid search below, we run a sparse grid of ($s$,$q$) values with the range of values for $s$ and $q$ informed by our analytical derivation of the planet parameters above. At each grid point, we find the best fit values of $\alpha$ and $\rho$ by minimizing $\chi^2$ using simple_fit and holding all other parameters fixed. We use the initial values of $\alpha$ and $\rho$ found above, and use the best fit single lens model values of $t_0$, $u_0$, and $t_E$. This works well for our case since we have a fairly good guess for $\alpha$ and the single lens parameters. If an initial estimate of $\alpha$ cannot be made (for example, when you have resonant or central caustic crossings or binary star events), then a good strategy would be to evaluate the $\chi^2$ for a number of $\alpha$ values uniformly sampled between $0^{\degree}$ and $360^{\degree}$ at each ($s$, $q$) grid point without fitting any parameter. If we don't have a good estimate of the single lens parameters either, then we can fit ($t_0$, $u_0$, $t_E$) at each ($s$, $q$, $\alpha$) grid point. 

The grid search module below is fairly flexible and can accomodate different grid search strategies. You can use the fit_params option to specify which (if any) parameters you want to fit at each grid point. If you provide a list of values for alpha_val, the code will run a loop to calculate $\chi^2$ values for each $\alpha$ at each grid point. Instead, specifying one $\alpha$ and adding "alpha" to the fit_params list will fit $\alpha$ at each grid point starting from the provided initial value. You can also parallelize this grid search on multiple cores, if available.



In [ ]:
"""
Binary Microlensing Grid Search Module (MulensModel Version)
=============================================================

This module performs a grid search over binary-lens microlensing parameters
(log10(s), log10(q), alpha) using parallel execution. For each grid point,
the user may choose which model parameters are optimized and which are held
fixed. If no parameters are selected for fitting, the chi^2 is evaluated
directly without running the optimizer.

Parallelization is handled via `multiprocess.Pool`, and progress tracking
is provided using `tqdm`.

Compatible with MulensModel Event objects.
"""

import numpy as np
import matplotlib.pyplot as plt
from multiprocess import Pool, cpu_count
from tqdm.auto import tqdm
import MulensModel as mm


# =============================================================================
# Helper functions
# =============================================================================

def _linear_tE_from_init(tE_init):
    """Accept either log10(t_E) or linear t_E, matching the previous heuristic."""
    return 10**tE_init if tE_init < 10 else tE_init


def _delta_chi2_from_event(event, n_fitted_params):
    """Use the same delta-chi^2 prescription as simple_fit."""
    event.fit_fluxes()
    chi2 = event.get_chi2()
    n_data = sum(len(dataset.time) for dataset in event.datasets)
    dof = n_data - n_fitted_params - 2
    return chi2, chi2 - dof


def _best_params_dict(fin_params, defaults, log_s_val, log_q_val):
    """Collect fitted and fixed parameters in one consistent result dict."""
    t_E = fin_params.get('t_E', defaults['t_E'])
    rho = fin_params.get('rho', defaults['rho'])
    alpha = fin_params.get('alpha', defaults['alpha'])

    return {
        't_0': fin_params.get('t_0', defaults['t_0']),
        'u_0': fin_params.get('u_0', defaults['u_0']),
        't_E': t_E,
        'rho': rho,
        's': fin_params.get('s', defaults['s']),
        'q': fin_params.get('q', defaults['q']),
        'alpha': alpha,
        'log_s': log_s_val,
        'log_q': log_q_val,
        'log_rho': np.log10(rho)
    }


# =============================================================================
# Worker function
# =============================================================================

def _grid_point_worker_mm(args):
    """
    Evaluate a single (log10(s), log10(q)) grid point using MulensModel.

    For a fixed (log_s, log_q) pair, this function either scans alpha values
    or fits alpha when `fit_params` contains 'alpha' and `alpha_vals` has one
    starting value. Any parameters in `fit_params` are optimized with
    `simple_fit`; all others are kept fixed. If `fit_params` is empty, chi^2
    is evaluated directly at each grid point without fitting.

    Parameters
    ----------
    args : tuple
        A packed tuple containing:
        - i, j : int
            Indices of the grid point in the log_s-log_q grid.
        - log_s_val : float
            log10 of binary separation (s).
        - log_q_val : float
            log10 of mass ratio (q).
        - alpha_vals : array-like
            Alpha grid values, or a single starting value when fitting alpha.
        - data_arrays : tuple
            (time, flux, err_flux) arrays for creating MulensData.
        - t0_init, u0_init, tE_init, log_rho_init : float
            Initial values for fixed parameters and local optimization.
        - fit_params : list[str]
            Parameters to fit. Empty means evaluate chi^2 without fitting.
        - bounds : dict or None
            Bounds passed to `simple_fit`.
        - maxfev : int
            Maximum function evaluations for optimizer.
        - magnification_methods : list
            Magnification method specification passed to
            ``mm.Model.set_magnification_methods``.

    Returns
    -------
    tuple
        (i, j, min_delta_chi2, best_alpha, best_params)
    """

    (i, j, log_s_val, log_q_val, alpha_vals, data_arrays,
     t0_init, u0_init, tE_init, log_rho_init,
     fit_params, bounds, maxfev, magnification_methods) = args

    time, flux, err_flux = data_arrays
    fit_params = [] if fit_params is None else list(fit_params)

    min_delta_chi2 = np.inf
    best_alpha = np.nan
    best_params = None
    t_E_init_linear = _linear_tE_from_init(tE_init)
    rho_init_linear = 10**log_rho_init

    for alpha_val in alpha_vals:
        # Create MulensData object
        data = mm.MulensData(
            data_list=[time, flux, err_flux],
            phot_fmt='flux'
        )
        
        # Create binary lens model with current grid point values
        model_params = {
            't_0': t0_init,
            'u_0': u0_init,
            't_E': t_E_init_linear,
            'rho': rho_init_linear,
            's': 10**log_s_val,
            'q': 10**log_q_val,
            'alpha': alpha_val
        }
        
        model = mm.Model(model_params)
        model.set_magnification_methods(magnification_methods)
        
        # Create Event
        event = mm.Event(datasets=[data], model=model)
        
        try:
            if fit_params:
                fin_params, fin_chi2, delta_chi2 = simple_fit(
                    event, fit_params, bounds=bounds, maxfev=maxfev
                )
            else:
                fin_chi2, delta_chi2 = _delta_chi2_from_event(event, n_fitted_params=0)
                fin_params = {}
            
            if delta_chi2 < min_delta_chi2:
                defaults = {
                    't_0': t0_init,
                    'u_0': u0_init,
                    't_E': t_E_init_linear,
                    'rho': rho_init_linear,
                    's': 10**log_s_val,
                    'q': 10**log_q_val,
                    'alpha': alpha_val
                }
                params = _best_params_dict(fin_params, defaults, log_s_val, log_q_val)
                min_delta_chi2 = delta_chi2
                best_alpha = params['alpha']
                best_params = params
        except Exception as e:
            # Skip failed fits/evaluations
            pass

    return (i, j, min_delta_chi2, best_alpha, best_params)


# =============================================================================
# Grid search driver
# =============================================================================

def grid_search_binary(event, log_s_vals, log_q_vals, alpha_vals,
                       t0_init, u0_init, tE_init, log_rho_init,
                       fit_params=None, bounds=None, verbose=False,
                       n_jobs=None, maxfev=500):
    """
    Perform a parallel grid search over binary microlensing parameters.

    The grid is defined in (log10(s), log10(q)). Alpha is normally scanned
    over `alpha_vals`; if `alpha_vals` contains exactly one value and
    `fit_params` contains 'alpha', that value is used as the initial alpha and
    alpha is optimized at each (log_s, log_q) grid point instead.

    For each (log_s, log_q) point on the grid, this function:
    1. Loops through all alpha values, unless alpha is being fitted
    2. Optimizes only the parameters listed in `fit_params`
    3. Evaluates chi^2 directly when `fit_params` is None or empty
    4. Stores the best result for that (log_s, log_q) point

    Parameters
    ----------
    event : mm.Event
        MulensModel Event object. Used to extract the photometry data and
        the magnification-method specification
        (via ``event.model.get_magnification_methods()``); the lens model
        itself is recreated per grid point.
    log_s_vals : array-like
        Grid values for log10(s)
    log_q_vals : array-like
        Grid values for log10(q)
    alpha_vals : array-like
        Grid values for alpha (degrees). If this has one value and
        `fit_params` includes 'alpha', the single value is used as the initial
        alpha for fitting instead of as a grid point.
    t0_init, u0_init, tE_init, log_rho_init : float
        Initial values for optimization or fixed-parameter evaluation.
        Note: tE_init should be log10(t_E) if fitting log_t_E.
    fit_params : list[str] or None, optional
        Parameters to fit with simple_fit. Use names accepted by simple_fit,
        e.g. ['t_0', 'u_0', 'log_t_E', 'log_rho'] or include 'alpha' when
        `alpha_vals` contains a single starting value. None or [] means no
        fitting; all parameters are fixed and chi^2 is evaluated directly.
    bounds : dict, optional
        Parameter bounds for simple_fit. Use MulensModel naming:
        e.g., {'t_0': (min, max), 'u_0': (min, max), 'log_t_E': (min, max),
        'alpha': (0, 360)}.
    verbose : bool, optional
        Print progress information
    n_jobs : int, optional
        Number of parallel processes. Defaults to cpu_count().
    maxfev : int, optional
        Maximum function evaluations per optimization. Default 500.

    Returns
    -------
    results : dict
        Contains:
        - 'log_s_vals': 1D array of log_s values
        - 'log_q_vals': 1D array of log_q values
        - 'delta_chi2_grid': 2D array of minimum delta_chi2 for each (log_s, log_q)
        - 'best_alpha_grid': 2D array of best alpha for each (log_s, log_q)
        - 'best_params_grid': 2D array of dicts with all best-fit params
        - 'sorted_results': List of dicts sorted by delta_chi2 (ascending)
        - 'fig': matplotlib figure with visualization
    """

    if n_jobs is None:
        n_jobs = cpu_count()

    alpha_vals = np.atleast_1d(alpha_vals).astype(float)
    fit_params = [] if fit_params is None else list(fit_params)

    if 'alpha' in fit_params and len(alpha_vals) != 1:
        raise ValueError(
            "To fit alpha, pass exactly one initial value in alpha_vals. "
            "Remove 'alpha' from fit_params to scan alpha on a grid."
        )

    # Extract data arrays from event (for pickling to workers)
    data = event.datasets[0]
    data_arrays = (np.array(data.time), np.array(data.flux), np.array(data.err_flux))

    # Extract magnification method specification from the parent event's model
    # so that each per-grid-point model is evaluated with the same method(s).
    try:
        magnification_methods = event.model.get_magnification_methods()
    except AttributeError:
        # Fallback for older MulensModel versions / un-configured models
        time_arr = data_arrays[0]
        magnification_methods = [time_arr.min(), 'VBM', time_arr.max()]

    # Allocate result arrays
    delta_chi2_grid = np.full((len(log_q_vals), len(log_s_vals)), np.inf)
    best_alpha_grid = np.full((len(log_q_vals), len(log_s_vals)), np.nan)
    best_params_grid = np.empty((len(log_q_vals), len(log_s_vals)), dtype=object)

    total_points = len(log_s_vals) * len(log_q_vals)

    if verbose:
        print(f"Processing {total_points} grid points using {n_jobs} processes...")
        if fit_params:
            print(f"Fitting parameters: {fit_params}")
        else:
            print("Fitting parameters: none; evaluating chi^2 directly")

    # Build task list
    tasks = []
    for i, log_s_val in enumerate(log_s_vals):
        for j, log_q_val in enumerate(log_q_vals):
            tasks.append((
                i, j, log_s_val, log_q_val, alpha_vals, data_arrays,
                t0_init, u0_init, tE_init, log_rho_init,
                fit_params, bounds, maxfev, magnification_methods
            ))

    # -------------------------------------------------------------------------
    # Parallel execution with progress bar
    # -------------------------------------------------------------------------

    results_list = []

    if n_jobs > 1:
        with Pool(processes=n_jobs) as pool:
            with tqdm(total=len(tasks),
                      desc="Binary grid search",
                      unit="grid",
                      disable=not verbose) as pbar:
                for res in pool.imap_unordered(_grid_point_worker_mm, tasks, chunksize=1):
                    results_list.append(res)
                    pbar.update(1)
    else:
        for task in tqdm(tasks, desc="Binary grid search",
                         unit="grid", disable=not verbose):
            results_list.append(_grid_point_worker_mm(task))

    # -------------------------------------------------------------------------
    # Populate result grids
    # -------------------------------------------------------------------------

    for i, j, min_dchi2, b_alpha, b_params in results_list:
        delta_chi2_grid[j, i] = min_dchi2
        best_alpha_grid[j, i] = b_alpha
        best_params_grid[j, i] = b_params

    if verbose:
        print(f"Grid search complete! Processed {total_points} points.")

    # -------------------------------------------------------------------------
    # Visualization
    # -------------------------------------------------------------------------

    log_s_grid, log_q_grid = np.meshgrid(log_s_vals, log_q_vals)

    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Handle potential inf/nan values for plotting
    plot_data = np.log10(np.clip(delta_chi2_grid, 1e-10, None))
    plot_data = np.where(np.isfinite(plot_data), plot_data, np.nan)
    
    im = ax.pcolormesh(
        log_s_grid, log_q_grid,
        plot_data,
        cmap='viridis',
        shading='auto'
    )

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label(r'$\log_{10}(\Delta \chi^2)$', fontsize=14)

    ax.set_xlabel('log10(s)', fontsize=14)
    ax.set_ylabel('log10(q)', fontsize=14)
    ax.set_title('Grid Search: Binary Lens Parameters', fontsize=16)

    # Find global minimum (excluding inf values)
    valid_mask = np.isfinite(delta_chi2_grid)
    if valid_mask.any():
        masked_grid = np.where(valid_mask, delta_chi2_grid, np.inf)
        min_idx = np.unravel_index(np.argmin(masked_grid), delta_chi2_grid.shape)
        ax.plot(log_s_grid[min_idx], log_q_grid[min_idx],
                'r*', markersize=20, label='Global minimum')

    ax.legend()
    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------------------------
    # Sorted results
    # -------------------------------------------------------------------------

    sorted_results = []
    for i in range(len(log_s_vals)):
        for j in range(len(log_q_vals)):
            if best_params_grid[j, i] is not None:
                sorted_results.append({
                    'log_s': log_s_vals[i],
                    'log_q': log_q_vals[j],
                    's': 10**log_s_vals[i],
                    'q': 10**log_q_vals[j],
                    'best_alpha': best_alpha_grid[j, i],
                    'delta_chi2': delta_chi2_grid[j, i],
                    'all_params': best_params_grid[j, i]
                })

    sorted_results.sort(key=lambda x: x['delta_chi2'])

    return {
        'log_s_vals': log_s_vals,
        'log_q_vals': log_q_vals,
        'delta_chi2_grid': delta_chi2_grid,
        'best_alpha_grid': best_alpha_grid,
        'best_params_grid': best_params_grid,
        'sorted_results': sorted_results,
        'fig': fig
    }


In [ ]:
# Run the grid search. Takes about ~ 8 mins to run on a Macbook Pro with 6 cores

# Define grid ranges
log_s_vals = np.linspace(np.log10(si)-0.2, np.log10(si)+0.7, 20)
log_q_vals = np.linspace(-4.25, -2.75, 20)
alpha_vals = alphai

# Initial values for optimization (from best fit single lens model)
# Note: tE_init should be log10(t_E) since we fit log_t_E
t0_init = t0i
u0_init = u0i
log_tE_init = np.log10(tEi)
log_rho_init = np.log10(rhoi)

# Run grid search using MulensModel Event
# Note: event_roman should be your MulensModel Event object with data loaded
results = grid_search_binary(
    event_roman, log_s_vals, log_q_vals, alpha_vals,
    t0_init, u0_init, log_tE_init, log_rho_init,
    fit_params=['alpha', 'log_rho'],
    bounds={
        'u_0': (u0i - 2, u0i + 2),
        'log_rho': (-7, -0.5),
        't_0': (t0i - tEi, t0i + tEi),
        'log_t_E': (np.log10(max(0.1, tEi - 20)), np.log10(tEi + 20))
    },
    verbose=True, n_jobs=1, maxfev=250
)

# Access the best results
print("\n" + "="*60)
print("GRID SEARCH RESULTS")
print("="*60)
valid_chi2 = results['delta_chi2_grid'][np.isfinite(results['delta_chi2_grid'])]
if len(valid_chi2) > 0:
    print(f"\nGlobal minimum delta_chi2: {np.min(valid_chi2):.2f}")

# Top 5 best solutions
print("\nTop 5 Best Solutions (sorted by delta_chi2):")
print("-" * 60)
for i, res in enumerate(results['sorted_results'][:5]):
    print(f"\nRank {i+1}:")
    print(f"  log_s = {res['log_s']:.3f}  (s = {res['s']:.3f})")
    print(f"  log_q = {res['log_q']:.3f}  (q = {res['q']:.6f})")
    print(f"  alpha = {res['best_alpha']:.1f} deg")
    print(f"  delta_chi2 = {res['delta_chi2']:.2f}")
    if res['all_params']:
        print(f"  All params: {res['all_params']}")

# Access the global best solution
if results['sorted_results']:
    best = results['sorted_results'][0]
    print("\n" + "="*60)
    print("BEST FIT PARAMETERS")
    print("="*60)
    print(f"s = {best['s']:.4f}")
    print(f"q = {best['q']:.6f}")
    print(f"alpha = {best['best_alpha']:.2f} degrees")
    if best['all_params']:
        print(f"t_0 = {best['all_params']['t_0']:.6f}")
        print(f"t_E = {best['all_params']['t_E']:.6f}")
        print(f"u_0 = {best['all_params']['u_0']:.6f}")
        print(f"rho = {best['all_params']['rho']:.6f}")
    print(f"Delta chi2 = {best['delta_chi2']:.2f}")

# Save grid search results to disk for later re-use / plotting without re-running.
# The figure object is not picklable across sessions, so we drop it before saving.
GRID_RESULTS_PATH = Path("grid_search_results.pkl")

results_to_save = {k: v for k, v in results.items() if k != 'fig'}
with open(GRID_RESULTS_PATH, "wb") as f:
    pickle.dump(results_to_save, f)

print(f"\nSaved grid search results to {GRID_RESULTS_PATH.resolve()}")


In [ ]:
# Load the saved grid search results and re-plot the delta_chi2 grid
# without having to re-run the (slow) grid search above.
import pickle
from pathlib import Path

GRID_RESULTS_PATH = Path("grid_search_results.pkl")

with open(GRID_RESULTS_PATH, "rb") as f:
    loaded_results = pickle.load(f)

log_s_vals = loaded_results['log_s_vals']
log_q_vals = loaded_results['log_q_vals']
delta_chi2_grid = loaded_results['delta_chi2_grid']

log_s_grid, log_q_grid = np.meshgrid(log_s_vals, log_q_vals)

fig, ax = plt.subplots(figsize=(10, 8))

plot_data = np.log10(np.clip(delta_chi2_grid, 1e-10, None))
plot_data = np.where(np.isfinite(plot_data), plot_data, np.nan)

im = ax.pcolormesh(
    log_s_grid, log_q_grid,
    plot_data,
    cmap='viridis',
    shading='auto'
)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label(r'$\log_{10}(\Delta \chi^2)$', fontsize=14)

ax.set_xlabel('log10(s)', fontsize=14)
ax.set_ylabel('log10(q)', fontsize=14)
ax.set_title('Grid Search: Binary Lens Parameters (loaded from file)', fontsize=16)

valid_mask = np.isfinite(delta_chi2_grid)
if valid_mask.any():
    masked_grid = np.where(valid_mask, delta_chi2_grid, np.inf)
    min_idx = np.unravel_index(np.argmin(masked_grid), delta_chi2_grid.shape)
    ax.plot(log_s_grid[min_idx], log_q_grid[min_idx],
            'r*', markersize=20, label='Global minimum')
    ax.legend()

plt.tight_layout()
plt.show()

# Quick sanity check on the top solutions from the saved file
print("Top 3 solutions from saved results:")
for i, res in enumerate(loaded_results['sorted_results'][:3]):
    print(f"  Rank {i+1}: s={res['s']:.3f}, q={res['q']:.2e}, "
          f"alpha={res['best_alpha']:.1f}, delta_chi2={res['delta_chi2']:.2f}, t0 = {res['all_params']['t_0']:.2f}, tE = {res['all_params']['t_E']:.2f}, u0 = {res['all_params']['u_0']:.2f}, rho = {10**res['all_params']['log_rho']:.2f})")

The grid search gave us two regions of the grid with good fits to the light curve. We can now start an MCMC run from any of these initial parameters. Though in reality, we would first want to create finer grids in the regions with small $\chi^2$ values to get better starting points for the MCMC runs. We will also restrict the MCMC to much stricter bounds since we have a rough idea of how the $\chi^2$ varies with planetary parameters now.

Let us now refine this solution with another MCMC run:

In [ ]:
#run MCMC on the best solution to find final refined solution
#filter dict to only include linear parameters
best_model_params = {k: v for k, v in results['sorted_results'][0]['all_params'].items() if k in ['t_0', 'u_0', 't_E', 'rho', 's', 'q', 'alpha']}
#Initialize event_roman with the best solution
event_roman.model = mm.Model(best_model_params)
params_to_fit = ['t_0', 'u_0', 't_E', 'log_rho', 'log_s', 'log_q', 'alpha']

# Choose bounds to restrict the search to the vicinity of the best solution
grid_mcmc1 = micro_mc(event_roman, params_to_fit, log_probability, bounds={'t_0': (t0i - tEi, t0i + tEi), 't_E': (tEi - 4, tEi + 4), 'u_0': (u0i - 0.1, u0i + 0.1), 'log_rho': (-5, -0.5), 'log_s': (results['sorted_results'][0]['all_params']['log_s']-0.05, results['sorted_results'][0]['all_params']['log_s']+0.0), 'log_q': (results['sorted_results'][0]['all_params']['log_q']-0.075, results['sorted_results'][0]['all_params']['log_q']+0.075)})

#Run MCMC analysis with all plots enabled
results = informed_guess_mcmc.perform_mcmc_analysis(steps=1000, walkers=25, param_scales=np.array([1e-1, 1e-1, 1e-2, 1e-1, 1e-2, 1e-2, 1e-1]),
                               verbose=True, plot_corner=True, plot_fit=True,
                               plot_traces=True, plot_convergence=False, n_threads=1,
                               pool_init_data={
                                   'data_file': 'dc1_186_W149.txt',
                                   'data_kwargs': {'phot_fmt': 'mag', 'chi2_fmt': 'flux', 'ephemerides_file': None},
                                   'model_params': params_binary,
                                   'mag_methods': [data_Roman_W149.time.min(), 'point_source_point_lens', 2458398.0, 'VBM', 2458401.0, 'point_source_point_lens', data_Roman_W149.time.max()],
                               },
                               save_chains='grid_search_mcmc')

The MLE solution is a fairly good fit to the light curve! The posteriors can be improved by running longer chains with more walkers.

 <style>
.exercise {
    background-color: #E0E0E0;
    border-left: 8px solid #808080;
    padding: 10px 0 10px 20px;  /* top, right, bottom, left */
    margin: 20px 5px;  
    box-sizing: border-box;  
}
.exercise h2 {
    color: #808080;
    font-size: 24px;
}
.exercise p{
    margin: 0 20px;  /* Adjust this value to add space after the paragraph */
}
</style>
<div class="exercise">
    <h2>Exercise 7</h2>
    <p>Start from a different grid point (pick one a little far away from the best solution, but which still has a decent chi^2) and run an MCMC. Compare the solution it finds with the one we found. </p> 
    <br>
</div>

## Group Project: Binary lens modelling

Choose a binary lens light curve from the sample of light curves provided. Identify the type of light curve and perturbation (planet-resonant caustic, planet-major image, planet-minor image, binary star(?)). Then, fit a single lens model to find the best fit single lens parameters. Based on the type of perturbation, decide the best approach to fit a binary lens model. For example, you can start with an analytical planet parameter estimate for non-resonant planet perturbations and then use that to inform the bounds of a grid search. If you have a resonant or central caustic perturbation, then you can perform a grid search directly by sampling parameters over expected ranges for the given topology.




## Additional Resources
- [VBMicrolensing reference implementation](https://github.com/VBBinaryLensing/VBBinaryLensing) used for the magnification routines in this notebook.
- [Gaudi (2012) microlensing review](https://ui.adsabs.harvard.edu/abs/2012ARA%26A..50..411G/abstract) covering binary-lens caustic topology.
- [Roman Research Nexus](https://roman.ipac.caltech.edu/nexus) — information about kernels, data access, and team spaces.
- [RGES-PIT Microlensing resources](https://rges-pit.org/outreach/) — minicourse videos and supplementary tutorials referenced in this session.


## About this Notebook

**Authors:** Arjun Murlidhar, Amber Malpas, Meet J. Vyas  
**Maintainers:** RGES-PIT Working Group 9  
**Last Updated:** 20 Nov 2025  
**Contact:** murlidhar.4@buckeyemail.osu.edu

## Citations

If you use `VBMicrolensing`, `emcee`, or this notebook for published research, please cite the
authors. Follow these links for more information about citing:

* [Citing `VBMicrolensing`](https://github.com/valboz/VBMicrolensing?tab=readme-ov-file#attribution)
* [Citing `emcee`](https://github.com/dfm/emcee#attribution)
* [Citing **Roman Microlensing Data Challenge 2026 Notebooks**](https://github.com/rges-pit/data-challenge-notebooks/blob/main/zenodo.txt)

### Citing Roman Microlensing Data Challenge 2026 Notebooks

If you use our notebooks in your project, please cite:

```
Malpas, A., Murlidhar, A., Vandorou, K., Kruszyńska, K., Crisp, A., & Vyas, M. (2026). Roman Microlensing Data Challenge 2026 Notebooks (v1.0.0). Zenodo. https://doi.org/10.5281/zenodo.18262183
```

**BibTeX:**
```bibtex
@software{malpas_2025_17806271,
  author       = {Malpas, Amber and Murlidhar, Arjun and Vyas, Meet and Vandorou, Katie and Kruszyńska, Katarzyna and Crisp, Ali},
  title        = {Roman Microlensing Data Challenge 2026 Notebooks},
  month        = dec,
  year         = 2026,
  publisher    = {Zenodo},
  version      = {v1.0.0},
  doi          = {10.5281/zenodo.18262183},
  url          = {https://doi.org/10.5281/zenodo.18262183}
}
```

<!-- Footer Start -->

<!-- Footer End -->